In [0]:
%sql
CREATE TABLE IF NOT EXISTS mba.trusted.f_inmet_chuva(
    id_estacao BIGINT COMMENT 'Chave substituta do fato clima diário',
    anomes int COMMENT 'Ano mes referencia do periodo de chuvas',
    chuva_mm DOUBLE COMMENT 'Precipitação total do dia, em mm'
)
USING DELTA
COMMENT 'Fato mensal de chuva por estação meteorológica, agregado a partir das leituras horárias. Granularidade: estação x mes. Fonte: INMET - BDMEP, via mba.raw.clima_inmet.'

In [0]:
from pyspark.sql import functions as F

spark.sql(f"""
            select CAST(date_format(a.data, 'yyyyMM')AS INT) AS anomes
                , b.id_estacao
                , coalesce(sum(precipitacao_total_mm),0) chuva_mm
            from mba.raw.clima_inmet a
            join mba.trusted.d_estacao_meterologica b on a.codigo_wmo =b.codigo_wmo
            group by all
          """).createOrReplaceTempView("stg_clima_diario")


In [0]:
%sql
MERGE INTO mba.trusted.f_inmet_chuva AS tgt
USING stg_clima_diario AS src
ON tgt.id_estacao = src.id_estacao and tgt.anomes = src.anomes

WHEN MATCHED THEN
    UPDATE SET
        tgt.chuva_mm = src.chuva_mm

WHEN NOT MATCHED THEN
    INSERT ( id_estacao, anomes, chuva_mm )
    VALUES ( src.id_estacao, src.anomes, src.chuva_mm );

In [0]:
dbutils.notebook.exit("Executed")

In [0]:
%sql
select * from mba.trusted.f_inmet_chuva a
join mba.trusted.d_estacao_meterologica b on a.id_estacao=b.id_estacao